In [1]:
## init mongo db and fiftyone connection
import os

# Define the URI to point to your manual process
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost:44123"

import fiftyone as fo

# Verify connection
print(fo.core.odm.database.get_db_conn()) 


You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
Database(MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone'), 'fiftyone')


In [2]:
import fiftyone.brain as fob
from sklearn.preprocessing import normalize
import plotly.express as px
import skdim
import random
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import ot
from sklearn.manifold import TSNE
import cv2
from fiftyone import ViewField as F
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import random

# ## renders plotly properly in a html instance. 
# import plotly.io as pio
# pio.renderers.default = "notebook"


In [3]:
## Load dataset and views from mongodb 
dataset = fo.load_dataset("dugong")

## load the views
nc_view = dataset.load_saved_view("New_Caledonia")
wp_view = dataset.load_saved_view("West_Papua")

In [4]:
view_GAM = wp_view.match(
    F("subregion")=="GAM"
)
view_FRIWEN = wp_view.match(
    F("subregion")=="FRIWEN"
)
view_MANTASANDY = wp_view.match(
    F("subregion")=="MANTASANDY"
)
view_UM = wp_view.match(
    F("subregion")=="UM"
)

print("Number of images per geographical location")
print(f"WP- GAM: {len(view_GAM)}")
print(f"WP - FRIWEN: {len(view_FRIWEN)}")
print(f"WP - MANTASANDY: {len(view_MANTASANDY)}")
print(f"WP -UM: {len(view_UM)}")
print(f"NC: {len(nc_view)}")
print(f"WP (total): {len(wp_view)}")


Number of images per geographical location


WP- GAM: 512
WP - FRIWEN: 779
WP - MANTASANDY: 3
WP -UM: 745
NC: 716
WP (total): 2039


In [12]:
dict_wp = {
    'um':view_UM,
    'gam':view_GAM,
    'mantasandy':view_MANTASANDY,
    'friwen':view_FRIWEN
}

for name, dd in dict_wp.items():
    ## count the number of objects
    print(f"Running:{name}")

    number_of_objects = dd.count("ground_truth.detections")
    print(f"Number of objects:{number_of_objects}")

    # Filter detections in the 'audit' field
    medium_complexity_bg = dd.filter_labels(
        "audit", 
        F("background_complexity") == "medium"
    )

    high_complexity_bg = dd.filter_labels(
        "audit", 
        F("background_complexity") == "high"
    )

    low_complexity_bg = dd.filter_labels(
        "audit", 
        F("background_complexity") == "low"
    )

    # Check how many detections matched
    print(f"Number of images as:")
    high = dd.match(F("background_complexity") == "high")
    medium = dd.match(F("background_complexity") == "medium")
    low = dd.match(F("background_complexity") == "low")
    print(f"high: {high.count()}")
    print(f"medium: {medium.count()}")
    print(f"low: {low.count()}")
    print('- - '*2)
    print(f"OBJECTS")
    print('high complexity',high_complexity_bg.count("audit.detections"))
    print("medium complexity",medium_complexity_bg.count("audit.detections"))
    print("low complexity",low_complexity_bg.count("audit.detections"))
    print('-'*30)

Running:um
Number of objects:1308
Number of images as:
high: 4
medium: 741
low: 0
- - - - 
OBJECTS
high complexity 6
medium complexity 1302
low complexity 0
------------------------------
Running:gam
Number of objects:997
Number of images as:
high: 299
medium: 61
low: 152
- - - - 
OBJECTS
high complexity 571
medium complexity 122
low complexity 304
------------------------------
Running:mantasandy
Number of objects:3
Number of images as:
high: 2
medium: 1
low: 0
- - - - 
OBJECTS
high complexity 2
medium complexity 1
low complexity 0
------------------------------
Running:friwen
Number of objects:1307
Number of images as:
high: 4
medium: 775
low: 0
- - - - 
OBJECTS
high complexity 0
medium complexity 1302
low complexity 0
------------------------------


## Strategy for Splitting between train and test 

### Per-Site Stratified Random Splitting

Per-Site:  each geographical island is an independent unit, to prevent spatial bias.

Stratified: maintaining the ratio of "High/Medium/Low" complexity in every set.

Random Splitting: stochastic selection to ensure generalizability.

In [ ]:
from fiftyone import ViewField as F
import fiftyone.utils.random as four

# create a combined key: e.g., "WP_UM_medium" or "WP_GAM_high"
# ensures the split respects both the location and the difficulty
dataset.set_values(
    "stratify_key",
    [f"{r}_{m}_{c}" for r, m, c in zip(
        dataset.values("region"), 
        dataset.values("subregion"), 
        dataset.values("background_complexity")
    )]
)

print("Unique strata created:", dataset.count_values("stratify_key"))

Unique strata created: {'WP_MANTASANDY_medium': 1, 'WP_MANTASANDY_high': 2, 'WP_GAM_high': 299, 'WP_GAM_medium': 61, 'NC_NC_low': 108, 'WP_GAM_low': 152, 'NC_NC_medium': 270, 'NC_NC_high': 338, 'WP_FRIWEN_medium': 775, 'WP_UM_high': 4, 'WP_FRIWEN_high': 4, 'WP_UM_medium': 741}


In [18]:
dataset.count_values("subregion")

{'MANTASANDY': 3, 'FRIWEN': 779, 'GAM': 512, 'NC': 716, 'UM': 745}

In [6]:
from sklearn.model_selection import train_test_split
import pandas as pd

## percentage of each train, val, test
train_size = 0.7
test_size = 0.2 
val_size = 0.1
random_state = 42

# 1. Define your target islands (skipping MANTASANDY)
islands_to_split = ['UM', 'GAM', 'FRIWEN']

# 2. Preparation: Clear old tags to start fresh
dataset.untag_samples(["train", "test", "val"])

for island in islands_to_split:
    # Get a view of just this island
    island_view = dataset.match(F("subregion") == island)
    
    ids = island_view.values("id")
    # Our strata is the complexity (high/medium/low)
    strata = island_view.values("background_complexity")
    
    # --- STEP 1: Split off the TEST set (20%) ---
    # Stratify ensures the 'high complexity' ratio stays the same
    train_val_ids, test_ids = train_test_split(
        ids, 
        test_size= test_size, 
        stratify=strata,
        shuffle=True, 
        random_state=random_state
    )
    
    # Get strata for the remaining 80% to split again
    train_val_strata = [s for i, s in zip(ids, strata) if i in train_val_ids]
    
    # Split remaining 80% into Train (70% total) and Val (10% total) ---
    # 0.125 * 0.8 = 0.1 (which is 10% of the original total)
    train_ids, val_ids = train_test_split(
        train_val_ids, 
        test_size= (val_size/(1-test_size)), 
        stratify=train_val_strata, 
        random_state=random_state
    )
    
    # 3. Apply the tags in FiftyOne
    dataset.select(train_ids).tag_samples("train")
    dataset.select(val_ids).tag_samples("val")
    dataset.select(test_ids).tag_samples("test")
    
    print(f"Island {island}: Train={len(train_ids)}, Test={len(test_ids)}, Val={len(val_ids)}")

# 4. Add New Caledonia to 'train' only
nc_view = dataset.match(F("region") == "NC")
nc_view.tag_samples("train")

Island UM: Train=521, Test=149, Val=75
Island GAM: Train=357, Test=103, Val=52
Island FRIWEN: Train=545, Test=156, Val=78


### Repeated Random Sub-sampling Validation

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# 2. Preparation: Clear old tags to start fresh
dataset.untag_samples(["train", "test", "val"])

## percentage of each train, val, test
train_size = 0.7
test_size = 0.2 
val_size = 0.1
random_state = 42
num_runs = 4
islands_to_split = ['UM', 'GAM', 'FRIWEN']

for run in range(1, num_runs + 1):
    tag_suffix = f"_run{run}"
    print(f"\n--- Generating Split for Run {run} ---")
    
    for island in islands_to_split:
        island_view = dataset.match(fo.ViewField("mission_name") == island)
        ids = np.array(island_view.values("id"))
        strata = np.array(island_view.values("background_complexity"))
        
        # 1. 20% Test
        train_val_ids, test_ids = train_test_split(
            ids,
            test_size=test_size,
            stratify=strata,
            random_state=random_state + run # Change seed per run
        )
        
        # Get strata for the remaining 80%
        remaining_strata = [s for i, s in zip(ids, strata) if i in train_val_ids]
        
        # 2. 10% Val (0.125 of the 0.8 remaining)
        train_ids, val_ids = train_test_split(
            train_val_ids,
              test_size=(val_size/(1-test_size)),
                stratify=remaining_strata, 
                random_state=random_state + run
        )
        
        # 3. Apply specific run tags
        dataset.select(train_ids).tag_samples(f"train{tag_suffix}")
        dataset.select(val_ids).tag_samples(f"val{tag_suffix}")
        dataset.select(test_ids).tag_samples(f"test{tag_suffix}")

    # 4. Add NC to every training run
    nc_view = dataset.match(fo.ViewField("region") == "NC")
    nc_view.tag_samples(f"train{tag_suffix}")



# Structure model

In [ ]:
import fiftyone as fo
import os

def export_for_training(dataset, export_dir, run_number=1):
    """
    Exports specific run tags to a YOLO-formatted directory.
    """
    tags = {
        "train": f"train_run{run_number}",
        "val": f"val_run{run_number}",
        "test": f"test_run{run_number}"
    }

    for split, tag in tags.items():
        view = dataset.match_tags(tag)
        
        # Export in YOLO format (Compatible with RT-DETR)
        view.export(
            export_dir=export_dir,
            dataset_type=fo.types.YOLOv5Dataset, # Standard format for modern detectors
            split=split,
            label_field="ground_truth", # Or "audit" if you prefer
        )
    print(f"Export for Run {run_number} completed at: {export_dir}")

# Usage:
export_path = "./training_data_run1"
export_for_training(dataset, export_path, run_number=1)


In [ ]:
from ultralytics import YOLO, RTDETR

def train_model(model_type, data_yaml, epochs=50):
    """
    Args:
        model_type: "YOLO" or "RTDETR"
        data_yaml: Path to the exported dataset.yaml
    """
    if model_type.upper() == "YOLO":
        # Using a modern YOLO backbone (e.g., YOLOv11)
        model = YOLO("yolo11n.pt") 
    elif model_type.upper() == "RTDETR":
        # Using the Real-Time DETR backbone
        model = RTDETR("rtdetr-l.pt")
    else:
        raise ValueError("Unsupported model type")

    # Training logic is identical for both!
    results = model.train(
        data=data_yaml,
        epochs=epochs,
        imgsz=640,
        device=0, # GPU
        project="Dugong_Thesis",
        name=f"{model_type}_Run1"
    )
    return model

# To start with RT-DETR as you requested:
trained_model = train_model("RTDETR", "./training_data_run1/dataset.yaml")

In [ ]:
# After training, load your best weights
model = RTDETR("./Dugong_Thesis/RTDETR_Run1/weights/best.pt")

# Select the test view to see how it performed on the unseen West Papua data
test_view = dataset.match_tags("test_run1")

# Apply the model to generate a new field: 'predictions_rtdetr'
test_view.apply_model(model, label_field="predictions_rtdetr")

# Launch the App to visualize the 'Audit' vs 'Predictions'
session = fo.launch_app(dataset)